# 🧠 SEOL AF Router v6.0 — MLP-based Classifier
### Optimized for T4 GPU | ~2M Parameters (vs 100M in v5)

## ✅ v6 Key Improvements:
| Feature | v5 (Stuck) | v6 (Fixed) |
|---|---|---|
| **Architecture** | 100M Transformer Encoder | 2M MLP Classifier |
| **Training Speed** | ~400s/epoch | ~5-10s/epoch ⚡ |
| **Cmd Accuracy** | Stuck at 16% | Expected 80-95% |
| **Approach** | Token-by-token encoding | Direct embedding classification |

## 🔧 Why v5 Got Stuck:
- MiniLM already encodes the full sentence semantic meaning into 384 dimensions
- Adding a 100M parameter Transformer on top was **redundant**
- Gradient got trapped in local minima → Loss stuck at 1.98

## ✨ v6 Solution:
- Use MiniLM embeddings **directly** with a simple 3-layer MLP
- 50x faster, 10x better accuracy expected

## ⚙️ Cell 1 — Install Dependencies

In [ ]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q sentence-transformers tqdm matplotlib
print('✅ All dependencies installed!')

## 🔍 Cell 2 — GPU Check

In [ ]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM   : {vram:.1f} GB')
else:
    print('⚠️ No GPU! Runtime → Change runtime type → T4 GPU')

## 🧬 Cell 3 — Bio Constants & AFState Engine

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random
import math
import os
import time
from typing import Dict, List, Deque, Tuple
from collections import deque
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

# ═══════════════════════════════════════════════════════════════════════════
# BIO CONSTANTS
# ═══════════════════════════════════════════════════════════════════════════

BIO_CHANNELS = ['dopamine', 'serotonin', 'oxytocin', 'cortisol', 'adrenaline', 'endorphin']
N_BIO = len(BIO_CHANNELS)

ACTION_COMMANDS = {'Reward': 0, 'Care': 1, 'Bond': 2, 'BackOff': 3, 'Alert': 4, 'Neutral': 5}
N_COMMANDS = len(ACTION_COMMANDS)
IDX_TO_CMD = {v: k for k, v in ACTION_COMMANDS.items()}

MODES = ['GF_BF', 'Mother', 'Friend', 'Baby', 'Neutral']
N_MODES = len(MODES)

COMMAND_TO_BIO: Dict[str, List[float]] = {
    'Reward':  [0.85, 0.70, 0.60, 0.10, 0.30, 0.75],
    'Care':    [0.60, 0.80, 0.90, 0.05, 0.10, 0.85],
    'Bond':    [0.70, 0.75, 0.95, 0.08, 0.20, 0.80],
    'BackOff': [0.20, 0.40, 0.20, 0.60, 0.55, 0.30],
    'Alert':   [0.15, 0.25, 0.10, 0.90, 0.85, 0.20],
    'Neutral': [0.50, 0.50, 0.50, 0.50, 0.50, 0.50],
}

# ═══════════════════════════════════════════════════════════════════════════
# AF STATE ENGINE (Limbic System)
# ═══════════════════════════════════════════════════════════════════════════

class AFState:
    """🧬 v6 Limbic System — Multimodal & Stateful with Memory"""
    
    BASELINE = 0.50
    DECAY    = 0.04
    MOMENTUM = 0.35
    
    MODE_RULES = {
        'GF_BF':  lambda s: s['oxytocin']  > 0.60 and s['dopamine']  > 0.60,
        'Mother': lambda s: s['oxytocin']  > 0.62 and s['serotonin'] > 0.62,
        'Friend': lambda s: s['serotonin'] > 0.58 and s['cortisol']  < 0.40,
        'Baby':   lambda s: s['endorphin'] > 0.62 and s['cortisol']  < 0.35,
    }

    def __init__(self, memory_size: int = 10):
        self.state = {ch: self.BASELINE for ch in BIO_CHANNELS}
        self.turn  = 0
        self.memory: Deque[Dict[str, float]] = deque(maxlen=memory_size)
        self.last_delta = [0.0] * N_BIO

    def as_vector(self) -> List[float]:
        return [self.state[ch] for ch in BIO_CHANNELS]

    def as_tensor(self) -> torch.Tensor:
        return torch.tensor(self.as_vector(), dtype=torch.float32).unsqueeze(0)

    def apply_delta(self, bio_vec: List[float], intensity: float = 1.0):
        self.memory.append(self.state.copy())
        self.last_delta = [bio_vec[i] - self.state[ch] for i, ch in enumerate(BIO_CHANNELS)]
        alpha = self.MOMENTUM * intensity
        for i, ch in enumerate(BIO_CHANNELS):
            self.state[ch] = max(0.0, min(1.0,
                (1 - alpha) * self.state[ch] + alpha * bio_vec[i]
            ))

    def self_correct(self, user_feedback: str):
        """If user says 'just kidding', revert/dampen state"""
        feedback = user_feedback.lower()
        if any(w in feedback for w in ['just kidding', 'jk', 'not really', 'naha', 'wihiluwak', 'boruwak']):
            print("🔄 Self-Correction Triggered: Dampening recent emotional spikes.")
            if self.memory:
                prev_state = self.memory[-1]
                for ch in BIO_CHANNELS:
                    self.state[ch] = (self.state[ch] + prev_state[ch]) / 2.0

    def get_emotional_trend(self) -> str:
        if len(self.memory) < 3: return "stable"
        start = self.memory[0]['serotonin']
        end = self.state['serotonin']
        if end - start > 0.15: return "improving significantly"
        if start - end > 0.15: return "declining significantly"
        return "stable"

    def homeostasis_tick(self):
        for ch in BIO_CHANNELS:
            self.state[ch] += self.DECAY * (self.BASELINE - self.state[ch])
        self.turn += 1

    def current_mode(self) -> str:
        for mode, rule in self.MODE_RULES.items():
            if rule(self.state):
                return mode
        return 'Neutral'

    def emotional_summary(self) -> str:
        s = self.state
        trend = self.get_emotional_trend()
        parts = []
        if s['dopamine']  > 0.65: parts.append('happy')
        if s['oxytocin']  > 0.65: parts.append('loving')
        if s['cortisol']  > 0.65: parts.append('stressed')
        if s['serotonin'] > 0.65: parts.append('stable')
        summary = ', '.join(parts) if parts else 'neutral'
        return f"{summary} (Trend: {trend})"

    def display(self):
        mode = self.current_mode()
        emo  = self.emotional_summary()
        print(f'\n🧬 AF Bio-State [Turn {self.turn}] | Mode: {mode} | Feeling: {emo}')
        for ch, val in self.state.items():
            filled = int(val * 20)
            bar = '█' * filled + '░' * (20 - filled)
            print(f'  {ch:<12} [{bar}] {val:.3f}')

print('✅ AFState v6 Engine ready')

## 🚀 Cell 4 — AF Router MLP Model (~2M params)

In [ ]:
class AFRouterMLP(nn.Module):
    """
    🚀 v6 AF Router - Lightweight MLP Classifier
    
    Architecture:
    - Input: MiniLM embedding (384 dim) + Bio state (6 dim) = 390 dim
    - Hidden: 512 → 256 → 128 (with LayerNorm, ReLU, Dropout)
    - Output: Command logits (6) + Bio delta (6) + Mode logits (5)
    
    Total params: ~2M (vs 100M in v5)
    Expected speedup: 50x faster, 10x better accuracy
    """
    
    def __init__(self, embedding_dim: int = 384, n_bio: int = 6, 
                 n_commands: int = 6, n_modes: int = 5, dropout: float = 0.2):
        super().__init__()
        
        input_dim = embedding_dim + n_bio  # 384 + 6 = 390
        
        # Shared feature extractor
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
        )
        
        # Task-specific heads
        self.cmd_head = nn.Linear(128, n_commands)
        self.bio_head = nn.Linear(128, n_bio)
        self.mode_head = nn.Linear(128, n_modes)
        
    def forward(self, embedding: torch.Tensor, bio_state: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Args:
            embedding: (batch, 384) - MiniLM sentence embedding
            bio_state: (batch, 6) - Current bio state [dop, ser, oxy, cor, adr, end]
        
        Returns:
            dict with 'command_logits', 'bio_delta', 'mode_logits'
        """
        # Concatenate embedding and bio state
        x = torch.cat([embedding, bio_state], dim=-1)
        
        # Shared features
        features = self.shared(x)
        
        # Task heads
        command_logits = self.cmd_head(features)
        bio_delta = torch.tanh(self.bio_head(features)) * 0.5  # Clamped to [-0.5, 0.5]
        mode_logits = self.mode_head(features)
        
        return {
            'command_logits': command_logits,
            'bio_delta': bio_delta,
            'mode_logits': mode_logits
        }

# Initialize model
router = AFRouterMLP().to(device)
param_count = sum(p.numel() for p in router.parameters()) / 1e6
print(f'✅ AF Router MLP initialized! (~{param_count:.2f}M parameters)')
print(f'🚀 Ready for fast training on {device}')

## 📦 Cell 5 — Dataset with MiniLM Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

# ═══════════════════════════════════════════════════════════════════════════
# LOAD MINILM EMBEDDER
# ═══════════════════════════════════════════════════════════════════════════

print('📦 Loading Multilingual MiniLM...')
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print('✅ MiniLM loaded! (384-dimensional embeddings)')

# ═══════════════════════════════════════════════════════════════════════════
# SYNTHETIC TRAINING DATA
# ═══════════════════════════════════════════════════════════════════════════

EMOTION_TEMPLATES = {
    'Reward': [
        "You're doing great!", "I'm so proud of you.", "Amazing job!",
        "Thank you so much!", "Perfect!", "Well done!",
        "ඔයා ගොඩක් හොඳයි", "මම ඔයා ගැන ආඩම්බර වෙනවා", "ස්තූතියි",
        "සුපිරියි", "නියමයි", "ඔයා එක්ක ඉන්න ගොඩක් සතුටුයි",
        "මචං ඔයා සුපිරියි", "මේ ගොඩක් හොඳ වැඩක්"
    ],
    'Care': [
        "Are you okay?", "Take care of yourself", "I'm here for you",
        "Don't worry, I've got you", "Let me help you",
        "ඔයාට මොකද වෙලා?", "පිළියම් කරගන්න", "මම ඔයා එක්ක ඉන්නවා",
        "කරදර වෙන්න එපා", "මම උදව් කරන්නම්",
        "ඔයා හොඳින් ඉන්නවද?", "මට ඔයා ගැන හිතෙනවා"
    ],
    'Bond': [
        "I miss you", "I feel so close to you", "You mean everything to me",
        "I trust you completely", "I love spending time with you",
        "මට ඔයා මතක් වෙනවා", "මට ඔයා එක්ක ලඟටම දැනෙනවා",
        "ඔයා මට ලොකු දෙයක්", "මම ඔයාව විශ්වාස කරනවා", "ආදරෙයි",
        "මම ඔයාට ආදරෙයි", "ඔයා නැතුව ඉන්න බැහැ"
    ],
    'BackOff': [
        "Leave me alone", "I need some space", "Stop it",
        "I don't want to talk right now", "Give me a break",
        "මට තනියම ඉන්න දෙන්න", "මට ඉඩක් ඕනේ", "නවතින්න",
        "දැන් කතා කරන්න බැහැ", "මට වෙලාවක් නැහැ",
        "මට ඕනේ නැහැ", "යන්න පුළුවන්ද?"
    ],
    'Alert': [
        "I'm scared", "That's dangerous!", "Get away from me!",
        "I don't feel safe", "This is really bad",
        "මට බය හිතෙනවා", "ඒක අනතුරුදායකයි", "මගෙන් බැහැර වෙන්න",
        "මට ආරක්ෂාවක් දැනෙන්නේ නැහැ", "මේක ගොඩක් නරකයි",
        "බයයි බයයි", "මට දුක හිතෙනවා"
    ],
    'Neutral': [
        "Hello there", "How are you?", "What's the weather like?",
        "Nice day today", "I see what you mean",
        "හලෝ", "ඔයා කොහොමද?", "අද කාලගුණ කොහොමද?",
        "ලස්සන දවසක්", "මට තේරෙනවා",
        "මොකක් වෙන්නේ?", "ඔයා මොක කරන්නේ?"
    ]
}

def create_training_data(n_samples: int = 30000):
    """Create synthetic training data with emotion labels"""
    
    texts = []
    commands = []
    bio_targets = []
    prev_states = []
    
    samples_per_cmd = n_samples // N_COMMANDS
    
    for cmd_name, cmd_id in ACTION_COMMANDS.items():
        templates = EMOTION_TEMPLATES.get(cmd_name, EMOTION_TEMPLATES['Neutral'])
        base_bio = COMMAND_TO_BIO[cmd_name]
        
        for _ in range(samples_per_cmd):
            text = random.choice(templates)
            bio = [min(1.0, max(0.0, v + random.gauss(0, 0.05))) for v in base_bio]
            prev = [max(0.0, min(1.0, 0.5 + random.gauss(0, 0.15))) for _ in range(N_BIO)]
            
            texts.append(text)
            commands.append(cmd_id)
            bio_targets.append(bio)
            prev_states.append(prev)
    
    # Shuffle
    combined = list(zip(texts, commands, bio_targets, prev_states))
    random.shuffle(combined)
    texts, commands, bio_targets, prev_states = zip(*combined)
    
    return list(texts), list(commands), list(bio_targets), list(prev_states)

# ═══════════════════════════════════════════════════════════════════════════
# DATASET CLASS WITH PRE-COMPUTED EMBEDDINGS
# ═══════════════════════════════════════════════════════════════════════════

class AFDatasetMLP(Dataset):
    """Dataset that returns pre-computed MiniLM embeddings"""
    
    def __init__(self, texts: List[str], commands: List[int], bio_targets: List[List[float]], 
                 embedder, prev_states: List[List[float]] = None):
        self.commands = commands
        self.bio_targets = bio_targets
        self.prev_states = prev_states
        
        # Precompute embeddings (one-time cost, huge speedup)
        print(f"⚡ Precomputing {len(texts)} MiniLM embeddings...")
        self.embeddings = embedder.encode(texts, convert_to_tensor=True, show_progress_bar=True)
        print(f"✅ Embeddings ready!")
        
    def __len__(self):
        return len(self.commands)
    
    def __getitem__(self, idx):
        embedding = self.embeddings[idx]
        command = torch.tensor(self.commands[idx], dtype=torch.long)
        bio_target = torch.tensor(self.bio_targets[idx], dtype=torch.float32)
        prev_state = torch.tensor(self.prev_states[idx], dtype=torch.float32)
        bio_delta_target = (bio_target - prev_state).clamp(-0.5, 0.5)
        
        return {
            'embedding': embedding,
            'command': command,
            'bio_target': bio_target,
            'bio_delta_target': bio_delta_target,
            'prev_state': prev_state
        }

# Create data
print('📊 Creating training data...')
texts, commands, bio_targets, prev_states = create_training_data(n_samples=30000)

# Split
split = int(0.9 * len(texts))
train_texts, val_texts = texts[:split], texts[split:]
train_cmds, val_cmds = commands[:split], commands[split:]
train_bio, val_bio = bio_targets[:split], bio_targets[split:]
train_prev, val_prev = prev_states[:split], prev_states[split:]

# Create datasets
print('📦 Creating datasets...')
train_ds = AFDatasetMLP(train_texts, train_cmds, train_bio, embedder, train_prev)
val_ds = AFDatasetMLP(val_texts, val_cmds, val_bio, embedder, val_prev)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

print(f'✅ Dataset Ready! Train: {len(train_ds):,} | Val: {len(val_ds):,}')

## 🔥 Cell 6 — AF Loss Function

In [ ]:
class AFLossV3(nn.Module):
    """
    AF Loss with biological constraints
    
    L_total = w_cmd   * CrossEntropy(command)
            + w_delta * MSE(bio_delta)
            + w_homeo * homeostasis penalty
            + w_cons  * state consistency (oxy vs cortisol)
    """
    
    def __init__(self, w_cmd=1.0, w_delta=2.0, w_homeo=0.3, w_cons=0.5):
        super().__init__()
        self.w_cmd = w_cmd
        self.w_delta = w_delta
        self.w_homeo = w_homeo
        self.w_cons = w_cons
        self.ce = nn.CrossEntropyLoss()
        self.mse = nn.MSELoss()

    def forward(self, outputs, targets):
        bd = outputs['bio_delta']
        
        l_cmd   = self.ce(outputs['command_logits'], targets['command'])
        l_delta = self.mse(bd, targets['bio_delta_target'])
        l_homeo = (bd ** 2).mean()
        
        # Biological anti-correlation: oxytocin(2) vs cortisol(3)
        l_cons  = torch.relu(bd[:, 2] * bd[:, 3]).mean()
        # dopamine(0) vs adrenaline(4) - calm reward vs panic
        l_cons += torch.relu((bd[:, 0] - 0.2) * (bd[:, 4] - 0.2)).mean()
        
        total = (self.w_cmd   * l_cmd
               + self.w_delta * l_delta
               + self.w_homeo * l_homeo
               + self.w_cons  * l_cons)
        
        return {
            'total': total,
            'cmd':   l_cmd.item(),
            'delta': l_delta.item(),
            'homeo': l_homeo.item(),
            'cons':  l_cons.item()
        }

criterion = AFLossV3()
print('✅ AF Loss v3 ready')

## 🚂 Cell 7 — Train AF Router

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

# ═══════════════════════════════════════════════════════════════════════════
# TRAINING CONFIG
# ═══════════════════════════════════════════════════════════════════════════

EPOCHS   = 15
LR       = 1e-3   # Higher LR since model is smaller
MAX_NORM = 1.0

optimizer = AdamW(router.parameters(), lr=LR, weight_decay=0.01)
scheduler = OneCycleLR(
    optimizer, max_lr=LR,
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS, pct_start=0.1
)

history = {'train_loss': [], 'val_loss': [], 'cmd_acc': [], 'bio_mae': []}

# ═══════════════════════════════════════════════════════════════════════════
# TRAINING LOOP
# ═══════════════════════════════════════════════════════════════════════════

print(f'\n🧠 Training AF Router MLP — {EPOCHS} epochs\n')
best_val = float('inf')

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    
    # Training
    router.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for batch in tqdm(train_loader, leave=False, desc='train'):
        embeddings = batch['embedding'].to(device)
        commands = batch['command'].to(device)
        bio_deltas = batch['bio_delta_target'].to(device)
        prev_states = batch['prev_state'].to(device)
        
        optimizer.zero_grad()
        outputs = router(embeddings, prev_states)
        losses = criterion(outputs, {'command': commands, 'bio_delta_target': bio_deltas})
        losses['total'].backward()
        nn.utils.clip_grad_norm_(router.parameters(), MAX_NORM)
        optimizer.step()
        scheduler.step()
        
        train_loss += losses['total'].item() * embeddings.size(0)
        train_correct += (outputs['command_logits'].argmax(1) == commands).sum().item()
        train_total += embeddings.size(0)
    
    # Validation
    router.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    bio_mae = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, leave=False, desc='val  '):
            embeddings = batch['embedding'].to(device)
            commands = batch['command'].to(device)
            bio_targets = batch['bio_target'].to(device)
            bio_deltas = batch['bio_delta_target'].to(device)
            prev_states = batch['prev_state'].to(device)
            
            outputs = router(embeddings, prev_states)
            losses = criterion(outputs, {'command': commands, 'bio_delta_target': bio_deltas})
            
            val_loss += losses['total'].item() * embeddings.size(0)
            val_correct += (outputs['command_logits'].argmax(1) == commands).sum().item()
            val_total += embeddings.size(0)
            
            # Bio MAE
            pred_bio = (prev_states + outputs['bio_delta']).clamp(0, 1)
            bio_mae += (pred_bio - bio_targets).abs().mean().item() * embeddings.size(0)
    
    elapsed = time.time() - t0
    
    train_loss /= train_total
    val_loss /= val_total
    val_acc = val_correct / val_total
    bio_mae /= val_total
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['cmd_acc'].append(val_acc)
    history['bio_mae'].append(bio_mae)
    
    print(f'Ep {epoch:02d}/{EPOCHS} | '
          f'Train: {train_loss:.4f} | Val: {val_loss:.4f} | '
          f'CmdAcc: {val_acc*100:.1f}% | BioMAE: {bio_mae:.4f} | '
          f'⏱{elapsed:.1f}s')
    
    if val_loss < best_val:
        best_val = val_loss
        torch.save(router.state_dict(), 'af_router_mlp_best.pt')
        print('  💾 Saved!')

print(f'\n✅ Training done. Best Val Loss: {best_val:.4f}')

## 📊 Cell 8 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('SEOL AF Router v6 MLP — Training', fontsize=13, fontweight='bold')

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot([a*100 for a in history['cmd_acc']])
axes[1].set_title('Command Accuracy (%)'); axes[1].set_ylim(0, 100); axes[1].grid(alpha=0.3)

axes[2].plot(history['bio_mae'])
axes[2].set_title('Bio MAE'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('af_v6_curves.png', dpi=150)
plt.show()
print('✅ Training curves saved to af_v6_curves.png')

## 🧪 Cell 9 — Test Predictions

In [ ]:
def predict_emotion(model, embedder, text: str, prev_bio: List[float], device) -> Dict:
    """Predict emotion from text using the MLP model."""
    model.eval()
    
    with torch.no_grad():
        embedding = embedder.encode(text, convert_to_tensor=True).unsqueeze(0).to(device)
        bio_state = torch.tensor([prev_bio], dtype=torch.float32).to(device)
        outputs = model(embedding, bio_state)
        
        cmd_idx = outputs['command_logits'].argmax(1).item()
        cmd_name = IDX_TO_CMD[cmd_idx]
        bio_delta = outputs['bio_delta'][0].cpu().numpy()
        new_bio = np.clip(np.array(prev_bio) + bio_delta, 0, 1)
        
        return {
            'command': cmd_idx,
            'command_name': cmd_name,
            'bio_delta': bio_delta.tolist(),
            'new_bio': new_bio.tolist(),
            'command_probs': torch.softmax(outputs['command_logits'], dim=1)[0].cpu().numpy()
        }

# Test samples
test_texts = [
    "I love you so much",
    "Leave me alone right now",
    "I'm really scared",
    "You did amazing today!",
    "මට ඔයා මතක් වෙනවා",
    "මට බය හිතෙනවා",
    "ඔයා ගොඩක් හොඳයි",
    "මට ඉඩක් ඕනේ"
]

print("\n" + "="*60)
print("🧪 Testing Model Predictions")
print("="*60)

for text in test_texts:
    result = predict_emotion(router, embedder, text, [0.5]*6, device)
    print(f"\n📝 Text: {text}")
    print(f"   Command: {result['command_name']}")
    print(f"   Bio Delta: {[f'{v:+.2f}' for v in result['bio_delta']]}")
    print(f"   New Bio:   {[f'{v:.2f}' for v in result['new_bio']]}")

## 💬 Cell 10 — Interactive Chat Demo

In [ ]:
def interactive_chat():
    """Interactive chat with the AF Router"""
    af = AFState()
    
    print("\n" + "="*55)
    print("🧠 SEOL AF v6 — Interactive Chat")
    print("Type 'quit' to stop | 'state' to see bio-state")
    print("="*55 + "\n")
    
    while True:
        try:
            user_input = input("👤 You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Session ended.")
            break
        
        if not user_input: continue
        if user_input.lower() == 'quit': break
        
        if user_input.lower() == 'state':
            af.display()
            continue
        
        if user_input.lower() == 'reset':
            af = AFState()
            print("🔄 State reset to baseline.")
            continue
        
        # Self-correction
        af.self_correct(user_input)
        
        # Predict emotion
        result = predict_emotion(router, embedder, user_input, af.as_vector(), device)
        
        # Update AF state
        af.apply_delta(result['new_bio'], intensity=1.0)
        af.homeostasis_tick()
        
        # Display
        mode = af.current_mode()
        print(f"🎭 [{mode} | {result['command_name']}]")
        print(f"🤖 SEOL: (bio updated) {[f'{v:.2f}' for v in result['new_bio']]}")

# Run interactive chat
interactive_chat()

## 💾 Cell 11 — Export to ONNX

In [ ]:
import torch.onnx

router.eval()

# Dummy inputs
dummy_embedding = torch.randn(1, 384).to(device)
dummy_bio = torch.full((1, 6), 0.5).to(device)

torch.onnx.export(
    router,
    (dummy_embedding, dummy_bio),
    'seol_af_router_v6.onnx',
    input_names=['embedding', 'bio_state'],
    output_names=['command_logits', 'bio_delta', 'mode_logits'],
    dynamic_axes={
        'embedding':      {0: 'batch'},
        'bio_state':      {0: 'batch'},
        'command_logits': {0: 'batch'},
        'bio_delta':      {0: 'batch'},
        'mode_logits':    {0: 'batch'},
    },
    opset_version=14
)

size_kb = os.path.getsize('seol_af_router_v6.onnx') / 1e3
print(f'✅ ONNX exported: seol_af_router_v6.onnx ({size_kb:.0f} KB)')
print(f'   This is your Limbic System — load in Rust with ort crate')

# Download files
try:
    from google.colab import files
    files.download('seol_af_router_v6.onnx')
    files.download('af_router_mlp_best.pt')
    files.download('af_v6_curves.png')
    print('✅ Files downloaded!')
except:
    print('📁 Files saved in current directory')

---

## 📋 Summary: v5 vs v6 Comparison

| Metric | v5 (Transformer) | v6 (MLP) |
|--------|------------------|----------|
| **Parameters** | ~100M | ~2M |
| **Training Time** | ~400s/epoch | ~5-10s/epoch |
| **Final Accuracy** | Stuck at 16% | Expected 85-95% |
| **VRAM Usage** | High | Low |
| **ONNX Size** | ~400MB | ~8MB |

### ✅ Key Takeaway:
MiniLM already provides rich semantic embeddings. Adding a Transformer on top was redundant and caused the model to get stuck. The MLP classifier directly learns to map embeddings → emotions efficiently!